In [10]:
# ------------------------------
# 📦 Imports
# ------------------------------
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime

In [11]:
# ===============================================================
# 🧩 FUNCTION: parse_number
# ===============================================================
def parse_number(value):
    """
    Converts values such as:
        "$20.00" -> 20.0
        "15.00"  -> 15.0
        "1,250"  -> 1250.0
    """

    if value is None:
        return None

    value = re.sub(r"[^\d.-]", "", value)

    try:
        return float(value)
    except ValueError:
        return None

In [17]:
# ===============================================================
# 🧩 FUNCTION: get_table
# ===============================================================
def get_table(url, debug=False):
    """
    Returns DataFrame:

    metric | avg | low | high
    """

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )
        response.raise_for_status()

    except Exception as e:
        print(f"Error fetching page: {e}")
        return pd.DataFrame(columns=["metric", "avg", "low", "high"])

    soup = BeautifulSoup(response.content, "html.parser")

    # Timestamp for this scrape
    scrape_datetime = datetime.now()

    # Last part of the URL
    url_name = url.rstrip("/").split("/")[-1]

    rows = soup.find_all("tr")

    if debug:
        print(f"Found {len(rows)} rows")

    output = []

    for row in rows:

        try:

            cells = row.find_all("td")

            # Ignore rows that aren't metric rows
            if len(cells) < 3:
                continue

            # First column
            metric = cells[0].get_text(strip=True)

            # Second column
            avg = parse_number(cells[1].get_text())

            # Third column
            price_range = cells[2]

            low_element = price_range.find("span", class_="barTextLeft")
            high_element = price_range.find("span", class_="barTextRight")

            low = (
                 parse_number(low_element.get_text(strip=True))
                 if low_element
                 else None
                 )

            high = (
                parse_number(high_element.get_text(strip=True))
                if high_element
                else None
                )

            output.append({
                "metric": metric,
                "avg": avg,
                "low": low,
                "high": high,
                "url": url,
                "url_name": url_name,
                "scrape_datetime": scrape_datetime
            })

        except Exception as e:

            if debug:
                print(f"Skipping row: {e}")

            continue

    df = pd.DataFrame(
        output,
        columns=[
            "metric",
            "avg",
            "low",
            "high",
            "url",
            "url_name",
            "scrape_datetime"
        ]
    )

    if debug:
        print(f"Extracted {len(df)} metrics")

    return df

In [ ]:
# ===============================================================
# ▶️ RUN
# ===============================================================

url = "https://www.numbeo.com/cost-of-living/in/Chicago"

df = get_table(
    url=url,
    debug=False
)

Found 93 rows
Extracted 56 metrics


In [19]:
df.head(56)

,metric,avg,low,high,url,url_name,scrape_datetime
0,,NaN,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
1,Meal at an Inexpensive Restaurant,20.00,15.00,35.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
2,Meal for Two at a Mid-Range Restaurant (Three ...,90.00,60.00,150.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
3,Combo Meal at McDonald's (or Equivalent Fast-F...,13.50,10.00,15.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
4,Domestic Draft Beer (0.5 Liter),7.00,4.00,10.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
5,Imported Beer (0.33 Liter Bottle),9.00,7.00,12.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
6,Cappuccino (Regular Size),5.48,4.00,8.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
7,"Soft Drink (Coca-Cola or Pepsi, 0.33 Liter Bot...",2.71,2.00,4.00,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
8,Bottled Water (0.33 Liter),2.30,1.50,3.50,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
9,"Milk (Regular, 1 Liter)",1.31,0.79,2.30,https://www.numbeo.com/cost-of-living/in/Chicago,Chicago,2026-07-30 16:07:57.226787
